### Read Data

In [8]:
import pandas as pd

df = pd.read_csv('../data/raw/raw_superstore.csv', encoding='ISO-8859-1')

#### Data Structure

In [9]:
print("Ukuran dataset:", df.shape)
print("\nKolom dataset:")
print(df.columns.tolist())

Ukuran dataset: (9994, 21)

Kolom dataset:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


#### Data Quality

In [10]:
print("\nMissing values:")
print(df.isna().sum())

print("\nJumlah baris duplikat:")
print(df.duplicated().sum())


Missing values:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64

Jumlah baris duplikat:
0


In [11]:
display(df.head())

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


### Data Cleaning

In [12]:
# Membuat salinan agar data mentah tetap aman
clean = df.copy()

# Menghapus spasi pada nama kolom
clean.columns = clean.columns.str.strip()

# Menghapus spasi pada kolom teks
for col in clean.select_dtypes(include="object").columns:
    clean[col] = clean[col].str.strip()

# Mengubah kolom tanggal
for col in ["Order Date", "Ship Date"]:
    clean[col] = pd.to_datetime(
        clean[col], errors="coerce"
    )

# Mengubah kolom numerik
for col in ["Sales", "Quantity", "Discount", "Profit"]:
    clean[col] = pd.to_numeric(
        clean[col], errors="coerce"
    )

# Memeriksa ulang kualitas data
print("Missing values setelah konversi:")
print(clean.isna().sum())

print("Ukuran data:", clean.shape)

Missing values setelah konversi:
Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
dtype: int64
Ukuran data: (9994, 21)


C:\Users\sadew\AppData\Local\Temp\ipykernel_1780\2103785447.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in clean.select_dtypes(include="object").columns:


### EDA

In [13]:

category_analysis = (
    clean.groupby("Category")
    .agg(
        total_sales=("Sales", "sum"),
        total_profit=("Profit", "sum"),
        total_quantity=("Quantity", "sum")
    )
    .reset_index()
)

category_analysis["profit_margin_pct"] = (
    category_analysis["total_profit"]
    / category_analysis["total_sales"]
    * 100
)

category_analysis = category_analysis.sort_values(
    "total_profit", ascending=False
)

display(category_analysis)


,Category,total_sales,total_profit,total_quantity,profit_margin_pct
2,Technology,836154.0330,145454.9481,6939,17.395712
1,Office Supplies,719047.0320,122490.8008,22906,17.035158
0,Furniture,741999.7953,18451.2728,8028,2.486695


In [14]:

monthly_analysis = (
    clean.dropna(subset=["Order Date"])
    .assign(
        month=lambda x: (
            x["Order Date"].dt.to_period("M").astype(str)
        )
    )
    .groupby("month")
    .agg(
        total_sales=("Sales", "sum"),
        total_profit=("Profit", "sum")
    )
    .reset_index()
)

display(monthly_analysis)


,month,total_sales,total_profit
0,2014-01,14236.8950,2450.1907
1,2014-02,4519.8920,862.3084
2,2014-03,55691.0090,498.7299
3,2014-04,28295.3450,3488.8352
4,2014-05,23648.2870,2738.7096
5,2014-06,34595.1276,4976.5244
6,2014-07,33946.3930,-841.4826
7,2014-08,27909.4685,5318.1050
8,2014-09,81777.3508,8328.0994
9,2014-10,31453.3930,3448.2573


### Save

In [16]:
clean.to_csv(
    "../data/processed/superstore_clean.csv",
    index=False
)

category_analysis.to_csv(
    "../data/processed/category_analysis.csv",
    index=False
)

monthly_analysis.to_csv(
    "../data/processed/monthly_analysis.csv",
    index=False
)

print("File hasil analisis berhasil disimpan.")


File hasil analisis berhasil disimpan.
